# Manual tests – Math Buddy blocks

> **Bu dosya teslime girmez.** Sadece blokları kendi girdilerinizle denemek için.
> Her bölümde girdileri değiştirip hücreyi tekrar çalıştırın (Shift + Enter).

**Önce aşağıdaki hücreyi bir kez çalıştırın:** ana notebook'taki (`math_buddy_its.ipynb`) bütün kodu sessizce
çalıştırır, böylece bütün modeller eğitilmiş olur (~5–10 saniye). Ana notebook'u değiştirirseniz bu hücreyi yeniden çalıştırın.

In [ ]:
# Load every model by running all code cells of the main notebook silently
import io, json, contextlib
import matplotlib.pyplot as plt

_original_show = plt.show
plt.show = lambda *args, **kwargs: plt.close("all")          # do not draw the main notebook's plots
with open("math_buddy_its.ipynb", encoding="utf-8") as f:
    _main = json.load(f)
with contextlib.redirect_stdout(io.StringIO()):               # hide the main notebook's prints
    for _cell in _main["cells"]:
        if _cell["cell_type"] == "code":
            exec("".join(_cell["source"]), globals())
plt.show = _original_show
print("All models loaded:",
      ", ".join(name for name in ["b3_model", "b5a_model", "b4_model", "b5b_model",
                                  "b1_model", "b2_model", "sentiment_model", "b3_mnist_model"] if name in globals()))

---
## B3 – El yazısı rakam

### Test 1: Test setinden bir resim seç
`DIGIT` ve `INDEX` değerlerini değiştirin. Bu resimler modelin eğitimde **hiç görmediği** resimler.
**Beklenen:** çoğu doğru (~%97). Özellikle **1** ve **8**'i deneyin; en çok bunlar karışıyordu.

In [ ]:
DIGIT = 8      # 0-9
INDEX = 0      # 0, 1, 2, ... (that digit's n-th image in the test set)

candidates = y_b3_test[y_b3_test == DIGIT].index
image_row = X_b3_test.loc[candidates[INDEX]]
result = b3_read_handwriting(image_row.to_numpy())

plt.figure(figsize=(2.2, 2.2))
plt.imshow(image_row.to_numpy().reshape(8, 8), cmap="gray_r")
plt.title(f"true {DIGIT} -> predicted {result['digit']}")
plt.axis("off")
plt.show()
print("B3 output:", result, "| CORRECT" if result["digit"] == DIGIT else "| WRONG")
print(f"(There are {len(candidates)} test images of digit {DIGIT}: INDEX can be 0 ... {len(candidates) - 1})")

### Test 2: Kendi rakamınızı "çizin"
8 satır × 8 karakter. `.` = boş, `+` = gri (orta mürekkep), `#` = siyah.
**Beklenen:** Dataset'teki rakamlar kalın ve kutuyu dolduracak şekilde yazılmış (B3 örnek resimlerine bakın).
İnce veya küçük çizerseniz model yanılabilir – bu, gerçek kullanımdaki bir riski gösterir.

In [ ]:
MY_DRAWING = '''
..####..
.#....#.
......#.
.....#..
....#...
...#....
..#.....
.######.
'''

INK = {".": 0, "+": 8, "#": 16}
rows = [line for line in MY_DRAWING.strip().splitlines()]
assert len(rows) == 8 and all(len(r) == 8 for r in rows), "The drawing must be exactly 8 lines of 8 characters"
pixels = np.array([[INK[ch] for ch in row] for row in rows], dtype=float)

result = b3_read_handwriting(pixels.ravel())
probabilities = b3_model.predict_proba(pd.DataFrame([pixels.ravel()], columns=pixel_columns))[0]
top3 = np.argsort(probabilities)[::-1][:3]

plt.figure(figsize=(2.2, 2.2))
plt.imshow(pixels, cmap="gray_r")
plt.title(f"predicted {result['digit']}")
plt.axis("off")
plt.show()
print("B3 output:", result)
print("Top-3:", {int(d): round(float(probabilities[d]), 3) for d in top3})

### Test 3: Modelin yanıldığı tüm test resimleri

In [ ]:
wrong = X_b3_test[b3_model.predict(X_b3_test) != y_b3_test]
fig, axes = plt.subplots(1, len(wrong), figsize=(1.6 * len(wrong), 2))
for ax, (index, row) in zip(np.atleast_1d(axes), wrong.iterrows()):
    ax.imshow(row.to_numpy().reshape(8, 8), cmap="gray_r")
    ax.set_title(f"true {y_b3_test[index]}\npred {b3_model.predict(row.to_frame().T)[0]}", fontsize=9)
    ax.axis("off")
plt.show()
print(f"{len(wrong)} wrong out of {len(X_b3_test)} test images")

### Test 4: Kendi çiziminiz – MNIST modeli (B3.9)
Paint / Önizleme / herhangi bir çizim programında **beyaz zemine siyah kalemle** bir rakam çizin, PNG olarak
`test_drawings/` klasörüne kaydedin ve `MY_DRAWING_FILE`'a adını yazın. Hücre iki modeli karşılaştırır.
**Beklenen:** MNIST modeli çoğu çizimi okur; 8×8 model çoğunu okuyamaz.

In [ ]:
MY_DRAWING_FILE = "test_drawings/drawn_six.png"

ink = load_ink(MY_DRAWING_FILE)
print("8×8 load_digits model:", b3_read_handwriting(drawing_to_8x8(ink).ravel()))
print("28×28 MNIST model    :", b3_read_drawing(ink))

fig, axes = plt.subplots(1, 3, figsize=(7, 2.5))
axes[0].imshow(ink, cmap="gray_r"); axes[0].set_title("your drawing", fontsize=9)
axes[1].imshow(drawing_to_8x8(ink), cmap="gray_r"); axes[1].set_title("what the 8×8 model sees", fontsize=9)
axes[2].imshow(drawing_to_mnist(ink), cmap="gray_r"); axes[2].set_title("what the MNIST model sees", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.show()

---
## B5a – Sesli rakam

### Test 1: Test setinden bir kayıt seç
Aşağıdaki listeden bir dosya adı seçip `FILE_NAME`'e yazın (bunlar eğitimde **kullanılmayan** kayıtlar).
**Beklenen:** çoğu doğru (~%91). 3↔6 ve 8→6 karışmaları olabilir.

In [ ]:
b5a_test_files = b5a_df.loc[X_b5a_test.index, "file_name"].sort_values().tolist()
print(f"{len(b5a_test_files)} test recordings, some examples:", b5a_test_files[::60])

FILE_NAME = "3_theo_14.wav" if "3_theo_14.wav" in b5a_test_files else b5a_test_files[0]
result = b5a_recognize_speech(os.path.join(FSDD_DIR, FILE_NAME))
true_digit = int(FILE_NAME[0])
print(f"\n{FILE_NAME}: true {true_digit} -> B5a output {result}",
      "| CORRECT" if result["digit"] == true_digit else "| WRONG")
if FILE_NAME not in b5a_test_files:
    print("Warning: this recording was used for TRAINING, so the test is not fair.")

### Test 2: Kendi sesiniz (isteğe bağlı)
1. Mac'te **QuickTime Player → File → New Audio Recording** ile İngilizce tek bir rakam söyleyin
   (örn. "seven"), kaydı proje klasörüne `my_seven.m4a` olarak kaydedin.
2. Terminalde proje klasöründe `.wav`'a çevirin (Mac'in kendi aracı, kurulum gerekmez):
   `afconvert -f WAVE -d LEI16@8000 my_seven.m4a my_seven.wav`
3. `MY_WAV` ve `MY_DIGIT`'i yazıp çalıştırın.

**Beklenen:** Sizin sesiniz model için **yeni bir ses** → doğruluk yaklaşık %50 (B5a.8'de ölçtüğümüz gibi).
Kaydın başında/sonunda uzun sessizlik olmaması ve sadece rakamın söylenmesi sonucu iyileştirir.

In [ ]:
MY_WAV = "my_seven.wav"
MY_DIGIT = 7

if os.path.exists(MY_WAV):
    result = b5a_recognize_speech(MY_WAV)
    print("B5a output:", result, "| CORRECT" if result["digit"] == MY_DIGIT else "| WRONG")
else:
    print(f"'{MY_WAV}' not found – record it first (see the steps above).")

---
## B4 – Dikkat

### Test 1: Kendi değerlerinizi girin
- `screen_gaze_ratio`: 0–1 (ekrana bakma oranı)
- `head_angle`: 0–60 derece
- `blink_rate`: dakikada göz kırpma (normal ~15, yorgun ~26)
- Bir değer yerine `None` yazarsanız "kamera yüzü kaybetti" durumunu denersiniz.

**Denemeniz gerekenler:** ekrana çok bakan ama çok göz kırpan biri (0.85, 5, 30) → yorgun mu?
Ekrana az bakan ama başı düz biri (0.4, 5, 15)?

In [ ]:
print(b4_detect_attention(screen_gaze_ratio=0.85, head_angle=5, blink_rate=30))
print(b4_detect_attention(screen_gaze_ratio=0.40, head_angle=5, blink_rate=15))
print(b4_detect_attention(screen_gaze_ratio=None, head_angle=10, blink_rate=15))

### Test 2: Bir değeri adım adım değiştir
Diğer iki değer sabitken `screen_gaze_ratio` 0'dan 1'e çıkınca kararın nerede değiştiğini görün.
`head_angle` ve `blink_rate`'i değiştirip tekrar deneyin.

In [ ]:
HEAD_ANGLE, BLINK_RATE = 10, 15
rows = []
for gaze in np.arange(0, 1.01, 0.1):
    out = b4_detect_attention(gaze, HEAD_ANGLE, BLINK_RATE)
    rows.append({"screen_gaze_ratio": round(gaze, 1), "state": out["state"], **out["probabilities"]})
print(pd.DataFrame(rows).to_string(index=False))

---
## B5b – Güven / tereddüt

### Test 1: Kendi değerlerinizi girin
- `speech_rate`: saniyede hece (emin ~4, tereddütlü ~3)
- `pause_count`: duraklama sayısı
- `response_delay`: cevaba başlamadan önceki saniye
- `pitch_variation`: ses tonundaki iniş çıkış (emin ~2, tereddütlü ~3)

**Denemeniz gerekenler:** hızlı konuşan ama 4 saniye geç başlayan biri? Geç başlayıp hiç duraklamayan biri?

In [ ]:
print(b5b_detect_confidence(speech_rate=4.2, pause_count=0, response_delay=4.0, pitch_variation=2.0))
print(b5b_detect_confidence(speech_rate=3.0, pause_count=0, response_delay=2.0, pitch_variation=2.5))

### Test 2: Gecikme arttıkça tereddüt olasılığı

In [ ]:
SPEECH_RATE, PAUSE_COUNT, PITCH_VARIATION = 3.5, 1, 2.5
for delay in [0.5, 1, 1.5, 2, 2.5, 3, 4, 5]:
    out = b5b_detect_confidence(SPEECH_RATE, PAUSE_COUNT, delay, PITCH_VARIATION)
    print(f"response_delay {delay:>4} s -> {out['state']:10s} p_hesitant = {out['p_hesitant']}")

---
## B1 – İlgi alanı + sentiment

Listeye kendi İngilizce mesajlarınızı ekleyin.
**Denemeniz gerekenler:**
- Eğitimde olmayan kelimeler: *"I played soccer"*, *"I went fishing"*, *"I ate sushi"*
- İki konulu mesajlar: *"I walked my dog and then we ate pizza"*
- Olumsuzlama: *"I don't hate football"*, *"football is not bad"*
- İroni: *"Oh great, football again"*
- Hiçbir ilgi alanı içermeyen mesaj: *"I slept all day"* – model yine de 4 sınıftan birini seçmek zorunda!

In [ ]:
MY_MESSAGES = [
    "I played soccer with my friends",
    "I walked my dog and then we ate pizza",
    "I don't hate football",
    "Oh great, football again",
    "I slept all day",
]
for message in MY_MESSAGES:
    print(f"{message!r:45s} -> {b1_analyze_message(message)}")

---
## B2 – Zorlanılan konu + sentiment

**Denemeniz gerekenler:** konu adını hiç söylemeyen mesajlar (*"What is 7 times 8?"*, *"How do I share 12 sweets
between 3 kids?"*), iki konulu mesajlar, Türkçe bir mesaj (model hiçbir kelimeyi tanımaz).

In [ ]:
MY_MESSAGES = [
    "What is 7 times 8?",
    "How do I share 12 sweets between 3 kids?",
    "I can do division but fractions are hard",
    "Çarpma çok zor",
]
for message in MY_MESSAGES:
    print(f"{message!r:45s} -> {b2_analyze_message(message)}")